# Check DV01 Deal Source Availability

This notebook checks whether deals listed in LMQR's `lmdv01/dv01_config.py` and `lmdv01/dv01_config_platf.py` have source data available.

Checks performed:

- Per-deal BigQuery tables: `mortgage_static_data_*`, `mortgage_history_data_*`, `mortgage_aggregated_monthly_data_*`
- Shared Figure Platform BigQuery tables
- Platform deal presence in Redshift `libremax.helocloandata`, which `dv01_update_platf.py` uses to build loan ID lists


In [9]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

LMQR_ROOT = Path(r"C:\Git\LMQR_prod")
if str(LMQR_ROOT) not in sys.path:
    sys.path.insert(0, str(LMQR_ROOT))

import lmdv01.dv01_config as dv01_cfg
import lmdv01.dv01_config_platf as platf_cfg
from lmdv01.dv01_utils import clean_deal_name
from lmdata.lmdb import executeSQLRedshiftDF

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = dv01_cfg.SA_FILE_PATH

PROJECT_ID = "dd-client-libremax"
DATASET_ID = "libremax"
BQ_TABLE_PREFIXES = {
    "static": "mortgage_static_data_",
    "history": "mortgage_history_data_",
    "monthly": "mortgage_aggregated_monthly_data_",
}
PLATFORM_SHARED_TABLES = [
    "mortgage_static_data_figure_platform",
    "mortgage_history_data_figure_platform",
    "mortgage_aggregated_monthly_data_figure_platform",
]


In [10]:
dv01_deals = pd.DataFrame({"config_source": "dv01_config", "deal_name": dv01_cfg.DV01_DEAL_NAME_LIST})
platform_deals = pd.DataFrame(
    {"config_source": "dv01_config_platf", "deal_name": platf_cfg.DV01_DEAL_NAME_LIST}
)

deals = pd.concat([dv01_deals, platform_deals], ignore_index=True)
deals["table_key"] = deals["deal_name"].map(clean_deal_name)
deals["pool_id"] = deals["deal_name"].str.replace(r"[^A-Za-z0-9]", "", regex=True)

# dv01_update.py always expects per-deal BQ tables. dv01_update_platf.py uses
# per-deal BQ tables for regular FIGRE deals, but GRADE / HE1A2 route through
# shared Figure Platform tables with loan IDs sourced from helocloandata.
deals["requires_per_deal_bq_tables"] = deals.apply(
    lambda row: row["config_source"] == "dv01_config"
    or (row["deal_name"].startswith("FIGRE") and row["deal_name"] != "FIGRE 2023-HE1A2"),
    axis=1,
)
deals["requires_helocloandata"] = deals.apply(
    lambda row: row["config_source"] == "dv01_config_platf"
    and (row["deal_name"].startswith("GRADE") or row["deal_name"] == "FIGRE 2023-HE1A2"),
    axis=1,
)

expected_tables = deals.loc[:, ["config_source", "deal_name", "table_key"]].merge(
    pd.DataFrame(
        [(table_type, f"{prefix}{{table_key}}") for table_type, prefix in BQ_TABLE_PREFIXES.items()],
        columns=["table_type", "table_template"],
    ),
    how="cross",
)
expected_tables["table_name"] = expected_tables.apply(
    lambda row: row["table_template"].format(table_key=row["table_key"]), axis=1
)

deals.sort_values(["config_source", "deal_name"]).reset_index(drop=True)


,config_source,deal_name,table_key,pool_id,requires_per_deal_bq_tables,requires_helocloandata
0,dv01_config,ACHM 2023-HE1,achm_2023_he1,ACHM2023HE1,True,False
1,dv01_config,ACHM 2023-HE2,achm_2023_he2,ACHM2023HE2,True,False
2,dv01_config,ACHM 2024-HE1,achm_2024_he1,ACHM2024HE1,True,False
3,dv01_config,ACHM 2024-HE2,achm_2024_he2,ACHM2024HE2,True,False
4,dv01_config,ACHM 2025-HE1,achm_2025_he1,ACHM2025HE1,True,False
5,dv01_config,FIGRE 2023-HE1,figre_2023_he1,FIGRE2023HE1,True,False
6,dv01_config,FIGRE 2023-HE2,figre_2023_he2,FIGRE2023HE2,True,False
7,dv01_config,FIGRE 2023-HE3,figre_2023_he3,FIGRE2023HE3,True,False
8,dv01_config,FIGRE 2024-HE1,figre_2024_he1,FIGRE2024HE1,True,False
9,dv01_config,FIGRE 2024-HE2,figre_2024_he2,FIGRE2024HE2,True,False


In [11]:
client = bigquery.Client(project=PROJECT_ID)

metadata_sql = f"""
SELECT
  table_id AS table_name,
  row_count,
  TIMESTAMP_MILLIS(creation_time) AS created_at,
  TIMESTAMP_MILLIS(last_modified_time) AS last_modified_at
FROM `{PROJECT_ID}.{DATASET_ID}.__TABLES__`
WHERE table_id IN UNNEST(@table_names)
"""

table_names = sorted(set(expected_tables["table_name"]) | set(PLATFORM_SHARED_TABLES))
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("table_names", "STRING", table_names)]
)
bq_tables = client.query(metadata_sql, job_config=job_config).result().to_dataframe()
bq_tables.head()


,table_name,row_count,created_at,last_modified_at
0,mortgage_aggregated_monthly_data_achm_2023_he1,0,2026-03-12 05:31:34.669000+00:00,2026-03-12 05:31:34.669000+00:00
1,mortgage_aggregated_monthly_data_achm_2023_he2,0,2026-03-12 05:43:50.722000+00:00,2026-03-12 05:43:50.722000+00:00
2,mortgage_aggregated_monthly_data_achm_2024_he1,0,2026-03-12 05:35:40.217000+00:00,2026-03-12 05:35:40.217000+00:00
3,mortgage_aggregated_monthly_data_achm_2024_he2,0,2026-03-12 05:35:00.990000+00:00,2026-03-12 05:35:00.990000+00:00
4,mortgage_aggregated_monthly_data_achm_2025_he1,0,2026-03-12 05:30:58.410000+00:00,2026-03-12 05:30:58.410000+00:00


In [12]:
table_status = expected_tables.merge(
    bq_tables[["table_name", "row_count", "last_modified_at"]], on="table_name", how="left"
)
table_status["exists"] = table_status["row_count"].notna()

bq_summary = (
    table_status.pivot_table(
        index=["config_source", "deal_name", "table_key"],
        columns="table_type",
        values="exists",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

for col in BQ_TABLE_PREFIXES:
    if col not in bq_summary.columns:
        bq_summary[col] = False

bq_summary["has_all_per_deal_bq_tables"] = bq_summary[list(BQ_TABLE_PREFIXES)].all(axis=1)
bq_summary = bq_summary.sort_values(["config_source", "deal_name"]).reset_index(drop=True)
bq_summary


,config_source,deal_name,table_key,history,monthly,static,has_all_per_deal_bq_tables
0,dv01_config,ACHM 2023-HE1,achm_2023_he1,True,True,True,True
1,dv01_config,ACHM 2023-HE2,achm_2023_he2,True,True,True,True
2,dv01_config,ACHM 2024-HE1,achm_2024_he1,True,True,True,True
3,dv01_config,ACHM 2024-HE2,achm_2024_he2,True,True,True,True
4,dv01_config,ACHM 2025-HE1,achm_2025_he1,True,True,True,True
5,dv01_config,FIGRE 2023-HE1,figre_2023_he1,True,True,True,True
6,dv01_config,FIGRE 2023-HE2,figre_2023_he2,True,True,True,True
7,dv01_config,FIGRE 2023-HE3,figre_2023_he3,True,True,True,True
8,dv01_config,FIGRE 2024-HE1,figre_2024_he1,True,True,True,True
9,dv01_config,FIGRE 2024-HE2,figre_2024_he2,True,True,True,True


In [13]:
platform_shared_status = (
    pd.DataFrame({"table_name": PLATFORM_SHARED_TABLES})
    .merge(bq_tables[["table_name", "row_count", "last_modified_at"]], on="table_name", how="left")
)
platform_shared_status["exists"] = platform_shared_status["row_count"].notna()

platform_deal_names = sorted(platform_deals["deal_name"].unique())
deal_list_sql = ", ".join("'" + deal.replace("'", "''") + "'" for deal in platform_deal_names)
helocloandata_sql = f"""
SELECT
    bbgdealname AS deal_name,
    COUNT(*) AS loan_history_rows,
    COUNT(DISTINCT loanid) AS loan_count,
    MIN(asofdate) AS min_asofdate,
    MAX(asofdate) AS max_asofdate
FROM libremax.helocloandata
WHERE bbgdealname IN ({deal_list_sql})
GROUP BY bbgdealname
"""

helocloandata_status = executeSQLRedshiftDF(db="lmdb", sql_query=helocloandata_sql)
helocloandata_status.columns = helocloandata_status.columns.str.lower()

platform_summary = platform_deals.merge(helocloandata_status, on="deal_name", how="left")
platform_summary["has_helocloandata"] = platform_summary["loan_history_rows"].notna()
platform_summary = platform_summary.sort_values("deal_name").reset_index(drop=True)

display(platform_shared_status)
platform_summary


,table_name,row_count,last_modified_at,exists
0,mortgage_static_data_figure_platform,0,2026-03-12 05:27:40.502000+00:00,True
1,mortgage_history_data_figure_platform,0,2026-03-12 05:27:41.434000+00:00,True
2,mortgage_aggregated_monthly_data_figure_platform,0,2026-03-12 05:27:43.080000+00:00,True


,config_source,deal_name,loan_history_rows,loan_count,min_asofdate,max_asofdate,has_helocloandata
0,dv01_config_platf,FIGRE 2023-HE1,NaN,NaN,NaN,NaN,False
1,dv01_config_platf,FIGRE 2023-HE1A2,104407.0,6884.0,2023-08-01,2026-02-01,True
2,dv01_config_platf,FIGRE 2023-HE2,NaN,NaN,NaN,NaN,False
3,dv01_config_platf,FIGRE 2023-HE3,NaN,NaN,NaN,NaN,False
4,dv01_config_platf,FIGRE 2024-HE1,34170.0,5985.0,2024-07-01,2024-12-01,True
5,dv01_config_platf,FIGRE 2024-HE2,NaN,NaN,NaN,NaN,False
6,dv01_config_platf,FIGRE 2024-HE3,NaN,NaN,NaN,NaN,False
7,dv01_config_platf,FIGRE 2024-HE4,NaN,NaN,NaN,NaN,False
8,dv01_config_platf,FIGRE 2024-HE5,NaN,NaN,NaN,NaN,False
9,dv01_config_platf,FIGRE 2024-HE6,4384.0,4384.0,2024-12-01,2024-12-01,True


In [14]:
final_summary = bq_summary.merge(
    deals[
        [
            "config_source",
            "deal_name",
            "requires_per_deal_bq_tables",
            "requires_helocloandata",
        ]
    ],
    on=["config_source", "deal_name"],
    how="left",
).merge(
    platform_summary[
        ["deal_name", "has_helocloandata", "loan_history_rows", "loan_count", "min_asofdate", "max_asofdate"]
    ],
    on="deal_name",
    how="left",
)

missing_shared_platform_tables = not platform_shared_status["exists"].all()

final_summary["notes"] = ""
final_summary.loc[
    final_summary["requires_per_deal_bq_tables"] & ~final_summary["has_all_per_deal_bq_tables"],
    "notes",
] = "Missing one or more required per-deal BigQuery tables"
final_summary.loc[
    final_summary["requires_helocloandata"] & final_summary["has_helocloandata"].ne(True),
    "notes",
] = "Missing required libremax.helocloandata rows"
final_summary.loc[
    final_summary["config_source"].eq("dv01_config_platf") & missing_shared_platform_tables,
    "notes",
] = final_summary["notes"].where(
    final_summary["notes"].eq(""), final_summary["notes"] + "; "
) + "Missing one or more shared Figure Platform BigQuery tables"

final_summary.sort_values(["config_source", "deal_name"]).reset_index(drop=True)


,config_source,deal_name,table_key,history,monthly,static,has_all_per_deal_bq_tables,requires_per_deal_bq_tables,requires_helocloandata,has_helocloandata,loan_history_rows,loan_count,min_asofdate,max_asofdate,notes
0,dv01_config,ACHM 2023-HE1,achm_2023_he1,True,True,True,True,True,False,NaN,NaN,NaN,NaN,NaN,
1,dv01_config,ACHM 2023-HE2,achm_2023_he2,True,True,True,True,True,False,NaN,NaN,NaN,NaN,NaN,
2,dv01_config,ACHM 2024-HE1,achm_2024_he1,True,True,True,True,True,False,NaN,NaN,NaN,NaN,NaN,
3,dv01_config,ACHM 2024-HE2,achm_2024_he2,True,True,True,True,True,False,NaN,NaN,NaN,NaN,NaN,
4,dv01_config,ACHM 2025-HE1,achm_2025_he1,True,True,True,True,True,False,NaN,NaN,NaN,NaN,NaN,
5,dv01_config,FIGRE 2023-HE1,figre_2023_he1,True,True,True,True,True,False,False,NaN,NaN,NaN,NaN,
6,dv01_config,FIGRE 2023-HE2,figre_2023_he2,True,True,True,True,True,False,False,NaN,NaN,NaN,NaN,
7,dv01_config,FIGRE 2023-HE3,figre_2023_he3,True,True,True,True,True,False,False,NaN,NaN,NaN,NaN,
8,dv01_config,FIGRE 2024-HE1,figre_2024_he1,True,True,True,True,True,False,True,34170.0,5985.0,2024-07-01,2024-12-01,
9,dv01_config,FIGRE 2024-HE2,figre_2024_he2,True,True,True,True,True,False,False,NaN,NaN,NaN,NaN,


In [15]:
missing_only = final_summary[final_summary["notes"].ne("")].sort_values(["config_source", "deal_name"])
missing_only.reset_index(drop=True)


,config_source,deal_name,table_key,history,monthly,static,has_all_per_deal_bq_tables,requires_per_deal_bq_tables,requires_helocloandata,has_helocloandata,loan_history_rows,loan_count,min_asofdate,max_asofdate,notes
0,dv01_config_platf,GRADE 2025-FIG6,grade_2025_fig6,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,Missing required libremax.helocloandata rows
1,dv01_config_platf,GRADE 2026-HB1,grade_2026_hb1,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,Missing required libremax.helocloandata rows
